# 🚀 正式提交版：洄瀾合成法（best method）

用我們本地驗證最強的**合成法**（公開 LB628 keep-Z 偵測 ＋ 我們新發明的雙向互惠連線 ＋ 物理對稱分裂閘）跑 **test set**、產生 `submission.csv`，拿到**真正的排行榜分數**。

> ✅ 純 `scipy`/`skimage`、**不需網路、不需 GPU**——符合 Code 競賽提交規則（Internet off、輸出 `submission.csv`）。
> 本地單樣本官方分 0.898（會比 LB 樂觀，真分數見排行榜）；目標：明顯贏過官方 baseline 0.143。

In [ ]:
import json, os
from collections import defaultdict
import blosc2
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter, distance_transform_edt
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max
try:
    from skimage.filters import threshold_otsu
except Exception:
    threshold_otsu = None

TEST_DIR = '/kaggle/input/competitions/biohub-cell-tracking-during-development/test'
SCALE = (1.625, 0.40625, 0.40625); SCALE_A = np.array(SCALE)
MAX_LINK_DISTANCE, DIV_DISTANCE, GAP_DISTANCE = 15.0, 8.0, 20.0
XY_DS, SMOOTH_SIGMA, MIN_PEAK_DIST, NMS_RADIUS_UM, REFINE_RZ, REFINE_RYX = 4, 1.0, 2, 2.65, 2, 5
DIV_COS_MAX = -0.2

def scaled_pairwise(A, B):
    d = A[:, None, :] - B[None, :, :]; return np.sqrt(((d * SCALE_A) ** 2).sum(axis=2))

def read_zarr_meta(zarr_path):
    with open(os.path.join(zarr_path, '0', 'zarr.json')) as f:
        meta = json.load(f)
    return tuple(meta['shape']), np.dtype(meta['data_type'])

def read_timepoint(zarr_path, t, shape, dtype):
    cp = os.path.join(zarr_path, '0', 'c', str(t), '0', '0', '0')
    if not os.path.exists(cp):
        return np.zeros(shape[1:], dtype=dtype)
    with open(cp, 'rb') as f:
        return np.frombuffer(blosc2.decompress(f.read()), dtype=dtype).reshape(shape[1:])
print('setup ready')

In [ ]:
# === 偵測（公開 LB628 keep-Z 古典；致謝 yusuketogashi/lb628-clean-room-no-gpu-baseline）===
def _block_mean_xy(vol, f=XY_DS):
    z, y, x = vol.shape; y2, x2 = (y // f) * f, (x // f) * f
    a = vol[:, :y2, :x2].astype(np.float32)
    return a.reshape(z, y2 // f, f, x2 // f, f).mean(axis=(2, 4))
def _normalize(ds):
    lo, hi = np.percentile(ds, 1.0), np.percentile(ds, 99.8)
    x = np.clip((ds - lo) / max(hi - lo, 1e-6), 0.0, 1.0)
    return np.clip(x - gaussian_filter(x, sigma=6.0), 0.0, None)
def _refine(vol, coord):
    zc, yc, xc = [int(round(v)) for v in coord]
    z0, z1 = max(0, zc-REFINE_RZ), min(vol.shape[0], zc+REFINE_RZ+1)
    y0, y1 = max(0, yc-REFINE_RYX), min(vol.shape[1], yc+REFINE_RYX+1)
    x0, x1 = max(0, xc-REFINE_RYX), min(vol.shape[2], xc+REFINE_RYX+1)
    patch = vol[z0:z1, y0:y1, x0:x1].astype(np.float32)
    if patch.size == 0: return coord
    w = np.clip(patch - np.percentile(patch, 20.0), 0.0, None); tot = float(w.sum())
    if tot <= 1e-6: return coord
    zz, yy, xx = np.indices(patch.shape, dtype=np.float32)
    return np.array([z0+float((zz*w).sum()/tot), y0+float((yy*w).sum()/tot), x0+float((xx*w).sum()/tot)])
def _physical_nms(coords, scores, radius_um):
    if len(coords) == 0: return coords
    cs = coords * SCALE_A; order = np.argsort(-scores); keep = []
    for i in order:
        if all(np.sqrt(((cs[i]-cs[j])**2).sum()) >= radius_um for j in keep): keep.append(i)
    return coords[keep]
def detect_combined(vol):
    sm = gaussian_filter(vol.astype(np.float32), sigma=(0.5, SMOOTH_SIGMA, SMOOTH_SIGMA))
    ds = _block_mean_xy(sm); norm = _normalize(ds)
    if norm.max() <= 0: return []
    thr = np.percentile(norm, 92.0)
    if threshold_otsu is not None:
        try: thr = max(thr, float(threshold_otsu(norm)))
        except Exception: pass
    peaks = peak_local_max(norm, min_distance=MIN_PEAK_DIST, threshold_abs=thr, exclude_border=False)
    if len(peaks) == 0: return []
    o = peaks.astype(np.float64); o[:, 1] = o[:, 1]*XY_DS + (XY_DS-1)/2.0; o[:, 2] = o[:, 2]*XY_DS + (XY_DS-1)/2.0
    refined = np.array([_refine(vol, c) for c in o])
    sc = np.array([vol[min(int(round(c[0])), vol.shape[0]-1), min(int(round(c[1])), vol.shape[1]-1), min(int(round(c[2])), vol.shape[2]-1)] for c in refined], dtype=np.float64)
    return [c for c in _physical_nms(refined, sc, NMS_RADIUS_UM)]
print('detection ready')

In [ ]:
# === 追蹤：雙向互惠連線 + 物理對稱分裂閘 + gap closing（我們的新發明）===
def track_novel(zarr_path, n_t, shape, dtype):
    nodes = {}; fids, fxyz = [], []; nid = 1
    for t in range(n_t):
        cents = detect_combined(read_timepoint(zarr_path, t, shape, dtype))
        ids, xyz = [], []
        for c in cents:
            nodes[nid] = (t, int(round(c[0])), int(round(c[1])), int(round(c[2]))); ids.append(nid); xyz.append(c); nid += 1
        fids.append(ids); fxyz.append(np.array(xyz) if xyz else np.empty((0, 3)))
    edges = []; has_in, out_count = set(), defaultdict(int)
    for t in range(n_t - 1):
        pid, pc = fids[t], fxyz[t]; cid, cc = fids[t+1], fxyz[t+1]
        if len(pid) == 0 or len(cid) == 0: continue
        D = scaled_pairwise(pc, cc); fwd = D.argmin(axis=1); bwd = D.argmin(axis=0)
        mp, mc, child_disp = set(), set(), {}
        for i in range(len(pid)):
            j = int(fwd[i])
            if int(bwd[j]) == i and D[i, j] <= MAX_LINK_DISTANCE:
                edges.append((pid[i], cid[j])); mp.add(i); mc.add(j); has_in.add(cid[j]); out_count[pid[i]] += 1; child_disp[i] = cc[j]-pc[i]
        for j in range(len(cid)):
            if j in mc: continue
            dd = np.sqrt((((pc-cc[j])*SCALE_A)**2).sum(1)); i = int(np.argmin(dd))
            if i in mp and dd[i] <= DIV_DISTANCE and out_count[pid[i]] < 2:
                v_new = (cc[j]-pc[i])*SCALE_A; v_old = child_disp[i]*SCALE_A
                cos = float(np.dot(v_new, v_old)/(np.linalg.norm(v_new)*np.linalg.norm(v_old)+1e-9))
                if cos < DIV_COS_MAX:
                    edges.append((pid[i], cid[j])); has_in.add(cid[j]); out_count[pid[i]] += 1
    has_out = set(out_count.keys())
    for t in range(n_t - 2):
        ends = [(i, n_) for i, n_ in enumerate(fids[t]) if n_ not in has_out]
        starts = [(j, n_) for j, n_ in enumerate(fids[t+2]) if n_ not in has_in]
        if not ends or not starts: continue
        ec = fxyz[t][[i for i, _ in ends]]; sc = fxyz[t+2][[j for j, _ in starts]]
        D = scaled_pairwise(ec, sc); rr, c2 = linear_sum_assignment(D)
        for ri, ci in zip(rr, c2):
            if D[ri, ci] <= GAP_DISTANCE:
                edges.append((ends[ri][1], starts[ci][1])); has_out.add(ends[ri][1]); has_in.add(starts[ci][1])
    return nodes, edges
print('tracking ready')

In [ ]:
# === 跑 test set → submission.csv ===
test_folders = sorted(d.replace('.zarr', '') for d in os.listdir(TEST_DIR) if d.endswith('.zarr'))
print('test 樣本數：', len(test_folders))
all_rows = []
for name in test_folders:
    zpath = os.path.join(TEST_DIR, name + '.zarr')
    shape, dtype = read_zarr_meta(zpath); n_t = shape[0]
    nodes, edges = track_novel(zpath, n_t, shape, dtype)
    for nid, (t, z, y, x) in nodes.items():
        all_rows.append({'dataset': name, 'row_type': 'node', 'node_id': nid,
                         't': t, 'z': z, 'y': y, 'x': x, 'source_id': -1, 'target_id': -1})
    for s, d in edges:
        all_rows.append({'dataset': name, 'row_type': 'edge', 'node_id': -1,
                         't': -1, 'z': -1, 'y': -1, 'x': -1, 'source_id': s, 'target_id': d})
    print(f'{name}: {len(nodes)} nodes, {len(edges)} edges')

submission = pd.DataFrame(all_rows)
submission.insert(0, 'id', range(len(submission)))
submission = submission[['id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']]
submission.to_csv('submission.csv', index=False)
print('\n已寫出 submission.csv：', len(submission), '列')
submission.head()